# Set microscope limits

This notebook writes the machine safety limits used by the Leica
navigator_expert driver. Limits are owned by navigator_expert.limits; no
controller or legacy motion.stage_config API is involved.

Edit the LIMITS table below, derive X/Y from four LAS X Point markers,
then use the final cell to publish limits.json into a date-stamped machine
ProgramData snapshot. Save this notebook before the final cell archives it
under a timestamped filename.

How to read the table:

- A range is an inclusive [minimum, maximum].
- An allowed list contains the exact permitted values.
- An empty list means reviewed and intentionally unrestricted.
- Every listed key is required; missing and unknown keys are rejected.
- The independent hardcoded physical stage backstop always applies.


In [1]:
import _bootstrap
from navigator_expert.limits.config import adopt_limits

LIMITS = {
    "x_um": {"range": [1000, 130000]},
    "y_um": {"range": [1000, 100000]},
    "z_galvo_um": {"range": [-250, 250]},
    "z_wide_um": {"range": [0, 8000]},
    "objective_slot": [],
    "set_zoom": [],
    "set_scan_speed": [],
    "set_scan_resonant": [],
    "set_scan_mode": [],
    "set_sequential_mode": [],
    "set_scan_field_rotation": [],
    "set_image_format": [],
    "set_z_stack_definition": [],
    "set_z_stack_step_size": [],
    "set_z_stack_size": [],
    "set_frame_accumulation": [],
    "set_frame_average": [],
    "set_line_accumulation": [],
    "set_line_average": [],
    "set_pinhole_airy": [],
    "set_detector_gain": [],
    "set_laser_intensity": [],
    "set_laser_shutter": [],
    "set_filter_wheel_slot": [],
    "set_filter_wheel_spectrum": [],
}

## Record X/Y limits from four LAS X points

In the active PythonInspect template in LAS X Navigator Expert, place exactly
four temporary Point markers at the safe sample boundaries. Do not leave scan
fields, focus markers, or other geometry in that template.

The cell below uses the Leica driver directly. It saves and preserves the
XML/RGN/LRP experiment trio first, reads exactly four clean Point markers
from that copy, updates LIMITS x_um and y_um to their inclusive bounding
rectangle, and leaves the active LAS X template unchanged.


In [2]:
import navigator_expert as drv
from navigator_expert.limits.adaptive import capture_adaptive_xy_limits

client = drv.connect_microscope(load_calibration=False)
captured_xy = capture_adaptive_xy_limits(client)
LIMITS.update(captured_xy["limits"])
print("Updated x_um:", LIMITS["x_um"])
print("Updated y_um:", LIMITS["y_um"])

Updated x_um: {'range': [1447.3179, 125469.3347]}
Updated y_um: {'range': [1431.7465, 81804.0864]}


## Save and publish

First save this notebook with Ctrl+S. Then run the final cell. It publishes
limits.json, archives the timestamped notebook under data/notebook, and
archives the saved LAS X .xml, .rgn, and .lrp files in the sibling
data/template folder as limits_<snapshot-datetime>.*. Publication fails
if that template trio is absent.


In [ ]:
from pathlib import Path

published = adopt_limits(
    LIMITS,
    notebook_paths=[_bootstrap.NOTEBOOK_PATH],
    template_paths=captured_xy["template_paths"],
)
template_dir = Path(published["snapshot"]) / "data" / "template"
archived_templates = list(template_dir.iterdir())
expected_stem = f"limits_{Path(published['snapshot']).name}"
archived_suffixes = {path.suffix.lower() for path in archived_templates}
if archived_suffixes != {".xml", ".rgn", ".lrp"} or any(
    path.stem != expected_stem for path in archived_templates
):
    raise RuntimeError(f"Incomplete template archive: {template_dir}")
print("Published:", published["limits_path"])
print("Archived notebook:", published["notebook_paths"][0])
print("Archived template:", template_dir)
published